# Multi-Band PQMF + Snake-Activation Conditional AudioGAN

A **100% time-domain** (zero-spectrogram) conditional GAN for spoken digits (AudioMNIST):
- **PQMF (Pseudo-Quadrature Mirror Filterbank):** Generator operates at 4 kHz across 4 time-domain subbands (`4 x 4096`), then a frozen near-perfect-reconstruction PQMF bank synthesizes crisp 16 kHz audio (`16,384` samples) — **4x faster** than full-rate 1D GANs.
- **SnakeBeta Activations:** $x + \frac{1}{\beta}\sin^2(\alpha x)$ gives 1D convolutions a built-in periodic bias for speech harmonics.
- **Multi-Period + Multi-Band Time-Domain Discriminators:** Catches pitch periodicity (periods `2, 3, 5, 7, 11`), subband coherence, and word envelope purely in the time domain, with **Projection Conditioning** for digits `0-9`.
- **Stability:** Hinge GAN loss + Spectral Norm + DiffAugment + EMA generator weights.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from pqmf_snake_gan import (
    PQMF,
    PQMFGenerator,
    CompositeDiscriminator,
    AudioMNISTDataset,
    build_dataset_cache,
)
from pqmf_snake_gan.train import (
    build_argparser,
    run_smoke_test,
    select_device,
    train,
    save_wav_pcm16,
)

device = select_device('auto')
print(f'PyTorch {torch.__version__} | Device: {device}')

## 1. Quick Smoke Test (PQMF SNR + Gradient Check)
Verifies >60 dB PQMF time-domain reconstruction and confirms gradients flow cleanly to **both** Generator and Discriminator (fixing the `.detach()` bug from earlier notebooks).

In [ ]:
run_smoke_test(device)

## 2. Inspect Dataset & PQMF 4-Subband Decomposition
Point `DATA_DIR` to either your raw AudioMNIST `.wav` folder (e.g. `AudioMNIST/data`) or your converted `.npy` folder (`audioMNIST`).

In [ ]:
DATA_DIR = 'audioMNIST'  # Folder of .wav or .npy files
CACHE_PATH = 'data_cache/audiomnist_16k.npz'  # Optional fast 16 kHz cache
SAMPLE_RATE = 16000

# Build one-time 16 kHz cache if it does not exist yet (speeds up training ~20-50x)
if not os.path.isfile(CACHE_PATH) and os.path.isdir(DATA_DIR):
    print('Building 16 kHz cache once...')
    build_dataset_cache(DATA_DIR, CACHE_PATH, target_sr=SAMPLE_RATE, audio_length=16384, npy_source_sr=48000)

data_source = CACHE_PATH if os.path.isfile(CACHE_PATH) else DATA_DIR
ds = AudioMNISTDataset(data_source, target_sr=SAMPLE_RATE, audio_length=16384, random_shift=False)
print(f'Loaded {len(ds):,} clips | Digit labels detected: {ds.has_labels}')

wav, label = ds[0]
pqmf = PQMF(subbands=4, taps=64)
with torch.no_grad():
    subbands = pqmf.analysis(wav.unsqueeze(0))[0]
    reconstructed = pqmf.synthesis(subbands.unsqueeze(0))[0]

fig, axes = plt.subplots(5, 1, figsize=(12, 8), sharex=False)
axes[0].plot(wav[0].numpy(), alpha=0.8, label='Original 16 kHz')
axes[0].plot(reconstructed[0].numpy(), alpha=0.6, linestyle='--', label='PQMF Reconstructed')
axes[0].set_title(f'Digit {label.item()} — Full 16 kHz Waveform vs PQMF Reconstruction')
axes[0].legend()
for b in range(4):
    axes[b + 1].plot(subbands[b].numpy(), linewidth=0.8)
    axes[b + 1].set_title(f'Time-Domain PQMF Subband {b} (4 kHz, band {b*2}-{(b+1)*2} kHz)')
plt.tight_layout()
plt.show()

display(Audio(wav[0].numpy(), rate=SAMPLE_RATE))

## 3. Train PQMF-Snake Conditional GAN
- **GPU (CUDA):** `base_channels=256`, `batch_size=64`, `epochs=50`
- **CPU only:** lower `base_channels=128`, `batch_size=16` for faster iteration

In [ ]:
OUT_DIR = 'runs/pqmf_snake'
args = build_argparser().parse_args([
    '--data-dir', data_source,
    '--out-dir', OUT_DIR,
    '--epochs', '50',
    '--batch-size', '64' if device.type == 'cuda' else '16',
    '--base-channels', '256' if device.type == 'cuda' else '128',
    '--sample-every', '5',
    '--save-every', '10',
    '--num-workers', '0',
])
train(args)

## 4. Generate & Listen to Digits `0-9` (Using EMA Generator)

In [ ]:
ckpt_path = os.path.join(OUT_DIR, 'checkpoint_latest.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt['config']

gen = PQMFGenerator(
    z_dim=cfg['z_dim'],
    num_classes=cfg['num_classes'],
    base_channels=cfg['base_channels'],
).to(device)
gen.load_state_dict(ckpt['ema']['shadow'])
gen.eval()

digits = list(range(10)) if cfg['num_classes'] > 0 else [0, 1, 2, 3]
with torch.no_grad():
    z = torch.randn(len(digits), cfg['z_dim'], device=device)
    lbl = torch.tensor(digits, dtype=torch.long, device=device) if cfg['num_classes'] > 0 else None
    waves = gen(z, lbl).squeeze(1).cpu().numpy()

for i, w in enumerate(waves):
    print(f'Digit {digits[i]}:' if cfg['num_classes'] > 0 else f'Sample {i}:')
    display(Audio(w, rate=cfg['sample_rate']))
    save_wav_pcm16(f'generated/pqmf_digit_{digits[i]}.wav', w, sample_rate=cfg['sample_rate'])